# 02465 Exam Cheatsheet Notebook

Fokus: DP, control i RL teme iz exam-instructions za **26 May 2026**. Notebook je napravljen kao brzi mix:

- tekst/formule koje treba znati objasniti na papiru,
- mini primjeri koje možeš ručno pratiti,
- kod obrasci iz `irlc` exercises/projects koje možeš adaptirati na ispitu.

Najvažniji mentalni model: DP/control/RL stalno rade istu stvar u različitim notacijama: **trenutni trošak/nagrada + procjena budućnosti**.

In [ ]:
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## 1. DP formulacija

Finite-horizon DP problem:

- stanje: `x_k in S_k`
- akcija: `u_k in A_k(x_k)`
- šum: `w_k ~ P_k(w | x_k, u_k)`
- dinamika: `x_{k+1} = f_k(x_k, u_k, w_k)`
- stage cost: `g_k(x_k, u_k, w_k)`
- terminal cost: `g_N(x_N)`

Za policy `pi = (mu_0, ..., mu_{N-1})`:

`J_{pi,k}(x) = E[ g_k(x, mu_k(x), w_k) + J_{pi,k+1}(x_{k+1}) ]`

Optimality equation:

`J_k*(x) = min_u E_w[ g_k(x,u,w) + J_{k+1}*( f_k(x,u,w) ) ]`

Ručno rješavanje: kreni od `J_N(x)=g_N(x)`, idi unazad, za svako stanje napravi tabelu `Q[u]`, izaberi minimum.

In [ ]:
# Mini DP by hand/code: deterministic inventory-like problem.
# x = current stock, u = ordered amount, w = demand.
# cost = holding + ordering + lost-sales proxy; terminal = leftover stock.

class TinyInventoryDP:
    def __init__(self, N=3, max_stock=3):
        self.N, self.max_stock = N, max_stock

    def S(self, k):
        return range(self.max_stock + 1)

    def A(self, x, k):
        return range(self.max_stock - x + 1)

    def Pw(self, x, u, k):
        return {0: 0.25, 1: 0.5, 2: 0.25}

    def f(self, x, u, w, k):
        return max(0, min(self.max_stock, x + u - w))

    def g(self, x, u, w, k):
        shortage = max(0, w - (x + u))
        next_stock = self.f(x, u, w, k)
        return 2*u + next_stock + 5*shortage

    def gN(self, x):
        return x


def DP_stochastic_local(model):
    J = [{} for _ in range(model.N + 1)]
    pi = [{} for _ in range(model.N)]
    J[model.N] = {x: model.gN(x) for x in model.S(model.N)}

    for k in range(model.N - 1, -1, -1):
        for x in model.S(k):
            Q = {}
            for u in model.A(x, k):
                Q[u] = sum(
                    pw * (model.g(x, u, w, k) + J[k+1][model.f(x, u, w, k)])
                    for w, pw in model.Pw(x, u, k).items()
                )
            pi[k][x] = min(Q, key=Q.get)
            J[k][x] = Q[pi[k][x]]
    return J, pi

model = TinyInventoryDP(N=3)
J, pi = DP_stochastic_local(model)
print("J[0] =", J[0])
print("pi[0] =", pi[0])

## 2. DPModel pattern iz course toolboxa

U `irlc.ex02.dp_model.DPModel` implementiraš:

- `f(x,u,w,k)`
- `g(x,u,w,k)`
- `gN(x)`
- `S(k)`
- `A(x,k)`
- `Pw(x,u,k)` kao dict `{w: probability}`

Tipični expected value u kodu:

```python
expected = sum(p * f(x) for x, p in distribution.items())
```

Policy i value su liste dictova:

- `J[k][x] = J_k(x)`
- `pi[k][x] = mu_k(x)`

In [ ]:
# Course implementation example if imports work in your environment.
from irlc.ex02.inventory import InventoryDPModel
from irlc.ex02.dp import DP_stochastic

model = InventoryDPModel(N=3)
J, pi = DP_stochastic(model)
print("Inventory DP: action at k=0, x=2:", pi[0].get(2))
print("Inventory DP: cost-to-go at k=0, x=2:", J[0].get(2))

## 3. Control: continuous/discrete LQ problems

Continuous linear-quadratic system:

`dx/dt = A x + B u + d`

running cost, common course convention:

`c(x,u) = 1/2 x^T Q x + 1/2 u^T R u + q^T x + r^T u + qc`

Discrete version:

`x_{k+1} = A_k x_k + B_k u_k + d_k`

Euler discretization:

`x_{k+1} = x_k + Delta f(x_k,u_k)`

For linear continuous dynamics this gives approximately:

`A_d = I + Delta A`, `B_d = Delta B`, `d_d = Delta d`.

Exponential integration for linear systems:

`A_d = exp(A Delta)` and `B_d u + d_d` comes from integrating `exp(A tau)(Bu+d)`.

In [ ]:
# Euler vs exact exponential discretization for harmonic oscillator.
from scipy.linalg import expm

A = np.array([[0, 1], [-1, 0]], dtype=float)
B = np.array([[0], [1]], dtype=float)
dt = 0.1

Ad_euler = np.eye(2) + dt * A
Bd_euler = dt * B
Ad_exact = expm(A * dt)

print("Euler Ad:\n", Ad_euler)
print("Exact Ad:\n", Ad_exact)
print("Euler Bd:\n", Bd_euler)

## 4. Discrete LQR

Problem:

`x_{k+1} = A_k x_k + B_k u_k + d_k`

Quadratic cost-to-go:

`J_k(x) = 1/2 x^T V_k x + v_k^T x + vc_k`

Optimal controller:

`u_k = L_k x_k + l_k`

DP idea is unchanged: initialize terminal quadratic cost, go backward, minimize a quadratic in `u` at each `k`.

In [ ]:
from irlc.ex05.dlqr import LQR, lqr_rollout

N = 20
A = np.array([[1, 1], [0, 1.]], dtype=float)  # double integrator-ish
B = np.array([[0], [1.]], dtype=float)
Q = np.diag([1., 0.])
R = np.array([[0.1]])

(L, l), (V, v, vc) = LQR(A=[A]*N, B=[B]*N, Q=[Q]*N, R=[R]*N, QN=Q)
x0 = np.array([[1.], [0.]])
xs, us = lqr_rollout(x0, A=[A]*N, B=[B]*N, d=None, L=L, l=l)

print("First feedback matrix L[0] =", L[0])
print("First action u0 =", us[0].ravel())
print("Final state approx =", xs[-1].ravel())

## 5. Linearization around `(xbar, ubar)`

For nonlinear discrete dynamics `x_{k+1}=f(x_k,u_k)`, linearize at point `(xbar, ubar)`:

`A = df/dx |_(xbar,ubar)`

`B = df/du |_(xbar,ubar)`

Affine correction:

`d = f(xbar, ubar) - A xbar - B ubar`

Then use LQR on:

`x_{k+1} approx A x_k + B u_k + d`

Course pattern: `model.f_jacobian(xbar, ubar)` and `model.f(xbar, ubar)`.

In [ ]:
from irlc.ex04.model_harmonic import DiscreteHarmonicOscilatorModel

model = DiscreteHarmonicOscilatorModel(dt=0.1)
xbar = np.array([0., 0.])
ubar = np.array([0.])
A, B = model.f_jacobian(xbar, ubar)
d = model.f(xbar, ubar) - A @ xbar - B @ ubar
print("A =\n", A)
print("B =\n", B)
print("d =", d)

## 6. PID control

Error relative to target `x*`:

`e_t = x* - x_t`

PID action:

`u_t = Kp e_t + Ki sum_tau e_tau Delta + Kd (e_t - e_{t-1}) / Delta`

Roles:

- `Kp`: reacts to current error, strong but can overshoot.
- `Ki`: removes steady-state bias, but can wind up.
- `Kd`: damping/prediction from error velocity, reduces oscillations.

For car steering/pendulum/harmonic oscillator: choose measured scalar error, often angle or position error; action is steering/force/torque.

In [ ]:
from irlc.ex04.pid import PID

pid = PID(dt=0.05, Kp=40, Ki=0, Kd=5, target=0)
for x in [1.0, 0.7, 0.2, -0.1]:
    print(f"x={x: .2f}, u={pid.pi(x): .2f}")

## 7. Trapezoid collocation for direct control

For continuous dynamics `dx/dt = f(x,u,t)`, direct collocation optimizes states/actions at grid points.

Trapezoid defect constraint between `k` and `k+1`:

`x_{k+1} - x_k - Delta/2 * ( f(x_k,u_k,t_k) + f(x_{k+1},u_{k+1},t_{k+1}) ) = 0`

Interpretation: both endpoints must agree with the dynamics on average. This turns an optimal control problem into nonlinear constraints + objective over all `x_k, u_k`.

In [ ]:
# Tiny trapezoid defect helper. At optimum this should be zero.
def trapezoid_defect(f, xk, uk, xkp1, ukp1, dt, tk=0.0):
    return xkp1 - xk - 0.5 * dt * (f(xk, uk, tk) + f(xkp1, ukp1, tk + dt))

# Example dynamics: xdot = v, vdot = u for state [x, v].
def double_integrator_f(x, u, t):
    return np.array([x[1], float(u[0])])

xk = np.array([0., 1.])
uk = np.array([0.])
xkp1 = np.array([0.1, 1.])
ukp1 = np.array([0.])
print("defect =", trapezoid_defect(double_integrator_f, xk, uk, xkp1, ukp1, dt=0.1))

## 8. Bandits

Bandit = one-state RL problem. At step `t` choose arm/action `A_t`, receive reward `R_t`. Goal: maximize expected reward / minimize regret.

Sample-average update:

`Q_{n+1}(a) = Q_n(a) + 1/N(a) * (R_n - Q_n(a))`

Nonstationary update with constant learning rate:

`Q_{t+1}(a) = Q_t(a) + alpha * (R_t - Q_t(a))`

Epsilon-greedy:

- probability `epsilon`: random action
- probability `1-epsilon`: greedy action

UCB:

`A_t = argmax_a [ Q_t(a) + c sqrt( log(t) / N_t(a) ) ]`

Higher `c` explores more.

In [ ]:
# Minimal bandit agents: sample average, constant alpha, UCB.
class EpsGreedyBandit:
    def __init__(self, k, epsilon=0.1, alpha=None):
        self.k, self.epsilon, self.alpha = k, epsilon, alpha
        self.Q = np.zeros(k)
        self.N = np.zeros(k)

    def act(self):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.k)
        return int(np.argmax(self.Q))

    def update(self, a, r):
        self.N[a] += 1
        step = self.alpha if self.alpha is not None else 1 / self.N[a]
        self.Q[a] += step * (r - self.Q[a])

class UCBBandit(EpsGreedyBandit):
    def __init__(self, k, c=2):
        super().__init__(k, epsilon=0)
        self.c = c
        self.t = 0

    def act(self):
        self.t += 1
        for a in range(self.k):
            if self.N[a] == 0:
                return a
        return int(np.argmax(self.Q + self.c * np.sqrt(np.log(self.t) / self.N)))

# quick stationary demo
true_q = np.random.randn(10)
agent = EpsGreedyBandit(k=10, epsilon=0.1)
for _ in range(1000):
    a = agent.act()
    r = true_q[a] + np.random.randn()
    agent.update(a, r)
print("best true arm:", np.argmax(true_q), "best learned arm:", np.argmax(agent.Q))

## 9. MDP notation and Bellman equations

MDP components:

- states `s in S`, actions `a in A(s)`
- transition/reward distribution `p(s', r | s, a)`
- discount `gamma`

Policy value:

`v_pi(s) = E_pi[ G_t | S_t=s ]`

Action value:

`q_pi(s,a) = E_pi[ G_t | S_t=s, A_t=a ]`

Bellman expectation:

`v_pi(s) = sum_a pi(a|s) sum_{s',r} p(s',r|s,a) [ r + gamma v_pi(s') ]`

Optimality:

`v*(s) = max_a sum_{s',r} p(s',r|s,a) [ r + gamma v*(s') ]`

Terminal states convention: future value is zero after terminal. Markov Reward Process = MDP with no action choice, or exactly one action per state.

In [ ]:
# Tiny MDP implemented using the course MDP interface.
from irlc.ex08.mdp import MDP

class TwoStateMDP(MDP):
    def __init__(self):
        super().__init__(initial_state="A")

    def A(self, s):
        return ["left", "right"] if s == "A" else ["wait"]

    def Psr(self, s, a):
        if s == "A" and a == "left":
            return {("T", 1): 1.0}
        if s == "A" and a == "right":
            return {("B", 0): 1.0}
        if s == "B":
            return {("T", 2): 1.0}
        return {}

    def is_terminal(self, s):
        return s == "T"

mdp = TwoStateMDP()
print("states:", mdp.states)
print("Psr(A,right):", mdp.Psr("A", "right"))

## 10. Value iteration and policy evaluation

Core helper:

`q(s,a) = sum_{s',r} p(s',r|s,a) [r + gamma V(s')]`

Policy evaluation uses expectation over policy:

`V(s) <- sum_a pi(a|s) q(s,a)`

Value iteration uses greedy backup:

`V(s) <- max_a q(s,a)`

If it converges, value iteration gives `v*` and greedy policy gives optimal policy.

In [ ]:
from irlc.ex08.value_iteration import value_iteration
from irlc.ex08.policy_evaluation import policy_evaluation

pi_star, V_star = value_iteration(mdp, gamma=0.9, theta=1e-10)
print("V*:", dict(V_star))
print("pi*:", pi_star)

random_pi = {s: {a: 1/len(mdp.A(s)) for a in mdp.A(s)} for s in mdp.nonterminal_states}
V_random = policy_evaluation(random_pi, mdp, gamma=0.9, theta=1e-10)
print("V_random:", dict(V_random))

## 11. Gridworld knobs: alpha, epsilon, gamma

For terminal reward only and otherwise zero reward:

- `gamma` near 1: distant terminal reward still matters; values propagate far.
- lower `gamma`: far reward is heavily discounted; values near terminal dominate.
- `epsilon` high: more exploration, noisier behavior, on-policy methods learn value of exploratory policy.
- `epsilon = 0`: greedy exploitation; can fail to discover better paths if Q initialized badly.
- `alpha` high: fast but noisy/unstable updates.
- `alpha` low: slow but smoother.

Algorithm distinctions:

- DP policy evaluation/value iteration: needs full model `p(s',r|s,a)`.
- MC: learns from complete episodes; no bootstrap.
- TD(0): learns `v_pi` by one-step bootstrap.
- Sarsa: on-policy control, learns `q_pi` for epsilon-greedy behavior.
- Q-learning: off-policy control, learns `q*` while behaving epsilon-greedy.
- Sarsa(lambda): Sarsa + eligibility traces; credit spreads backward along recent state-actions.

## 12. TD(0), Sarsa, Q-learning, MC, TD-lambda updates

TD(0) value prediction:

`V(S_t) <- V(S_t) + alpha [ R_{t+1} + gamma V(S_{t+1}) - V(S_t) ]`

Sarsa:

`Q(S,A) <- Q(S,A) + alpha [ R + gamma Q(S',A') - Q(S,A) ]`

Q-learning:

`Q(S,A) <- Q(S,A) + alpha [ R + gamma max_a Q(S',a) - Q(S,A) ]`

MC return update:

`Q(S_t,A_t) <- Q(S_t,A_t) + alpha [ G_t - Q(S_t,A_t) ]`

Sarsa(lambda):

`delta = R + gamma Q(S',A') - Q(S,A)`

`E(S,A) += 1`

For all `(s,a)`: `Q(s,a) += alpha delta E(s,a)`, `E(s,a) *= gamma lambda`.

In [ ]:
# Copy-paste friendly tabular update snippets.
Q = defaultdict(float)
V = defaultdict(float)
E = defaultdict(float)
gamma, alpha, lamb = 0.9, 0.1, 0.8
s, a, r, sp, ap, done = "s", "a", 1.0, "sp", "ap", False
actions_sp = ["ap", "other"]

# TD(0)
V[s] += alpha * (r + gamma * (0 if done else V[sp]) - V[s])

# Sarsa
Q[s, a] += alpha * (r + gamma * (0 if done else Q[sp, ap]) - Q[s, a])

# Q-learning
best_next = max(Q[sp, aa] for aa in actions_sp)
Q[s, a] += alpha * (r + gamma * (0 if done else best_next) - Q[s, a])

# Sarsa(lambda)
delta = r + gamma * (0 if done else Q[sp, ap]) - Q[s, a]
E[s, a] += 1
for key in list(E.keys()):
    Q[key] += alpha * delta * E[key]
    E[key] *= gamma * lamb

print("updates ran")

## 13. First-visit vs every-visit MC

Episode format: `(s_t, a_t, r_{t+1})`.

Return:

`G_t = r_{t+1} + gamma r_{t+2} + gamma^2 r_{t+3} + ...`

- first-visit MC: update only the first time `(s,a)` appears in the episode.
- every-visit MC: update every occurrence.

MC needs terminating episodes or artificial time limits, otherwise returns may never be available.

In [ ]:
def mc_returns_sa(episode, gamma=1.0, first_visit=True):
    # episode = [(s, a, r_next), ...]
    G = 0
    out = []
    for t in reversed(range(len(episode))):
        s, a, r = episode[t]
        G = r + gamma * G
        sa = (s, a)
        if (not first_visit) or (sa not in [(x[0], x[1]) for x in episode[:t]]):
            out.append((sa, G))
    return list(reversed(out))

episode = [("A", "right", 0), ("B", "wait", 2)]
print(mc_returns_sa(episode, gamma=0.9, first_visit=True))

## 14. On-policy/off-policy and terminating/non-terminating

On-policy: learns value of the same policy it uses to act.

- Sarsa learns value of epsilon-greedy behavior policy.
- MC on-policy control learns the epsilon-greedy policy it samples from.

Off-policy: learns target policy while behaving differently.

- Q-learning behaves epsilon-greedy but target backup is greedy, so it learns `q*` under standard assumptions.

Terminating tasks: episodes end, MC is natural.

Continuing/non-terminating tasks: use discounting, average reward formulations, or artificial truncation. TD methods can update online without waiting for episode end.

## 15. Project/exercise anchors

Use these as “where did I see this?” map:

- Project 1 / ex01-ex02: Pacman/kiosk/inventory, DP formulation and dictionaries.
- Project 2 / ex04-ex06: R2D2/Yoda, discretization, LQR, linearization, iLQR flavor.
- Project 3 / ex08-ex10: Jar-Jar, MDPs, Bellman equations, MC/TD/Sarsa/Q-learning.
- Project 4 / ex11-ex12: function approximation/rebels context, lambda traces and more advanced control/RL ideas.
- Exam folders: `irlc/exam/exam2023spring`, `exam2024spring`, `exam2025spring`, `exam2026spring`.

In [ ]:
# Jar-Jar project 3 check: exact q-values from your hand-in code.
from irlc.project3.jarjar import Q_exact, Q0_approximate, pi_optimal

gamma = 0.8
print("pi_optimal(-2), pi_optimal(2):", pi_optimal(-2), pi_optimal(2))
print("Q_exact(0, 1):", Q_exact(0, 1, gamma))
print("Q0_approximate(0.8, N=10):", Q0_approximate(gamma, 10))

## 16. Exam quick checklist

Can you do these without looking?

1. Given a tiny DP, write `S_k`, `A_k(x)`, `f_k`, `g_k`, `g_N`, `P(w|x,u)` and run the backward recursion by hand.
2. Convert a short MDP story/diagram into `p(s',r|s,a)` equations.
3. Write Bellman expectation and optimality equations for that MDP.
4. Explain which algorithm computes `v_pi`, `q_pi`, `v*`, or `q*`.
5. For a terminal-reward gridworld, predict the effect of changing `alpha`, `epsilon`, `gamma`.
6. Discretize `dx/dt = f(x,u)` with Euler and identify `A`, `B`, `Q`, `R` for LQR.
7. Linearize `f(x,u)` around `(xbar, ubar)` and compute affine term `d`.
8. Write PID action and explain `x*`, `Kp`, `Ki`, `Kd`.
9. Write trapezoid collocation defect constraint.
10. Implement dictionary expected values and tabular updates without hesitation.